# Real-network structural benchmark

This notebook constructs the empirical reference used to evaluate the LLM-generated networks.

- Directed nominations are represented as both **Mutual** (both arcs required) and **OR** (either arc sufficient) projections.
- The primary benchmark uses the Mutual projection and 57 friendship/friendship-like networks from 13 studies.
- Facebook ego networks are excluded from the primary benchmark because they are sampled ego neighborhoods, not complete bounded networks.
- Advice, cowork, instrumental, and discussion relations are retained only for the all-relations sensitivity analysis.
- Every study receives equal total weight in the primary plausibility ranges.
- Twelve raw self-nominations (four PEAS and eight MSW) are removed and recorded in the inventory.
- Seven of Chang et al.'s eight reference networks are available locally and flagged explicitly; the missing network is Moreno freshmen.
- Density is descriptive; the five plausibility metrics are average degree, clustering, LCC share, normalized path length, and modularity.


## 0 · Imports & Configuration

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
from matplotlib.lines import Line2D
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

REAL_NETWORK_DIR = BASE_DIR / "data" / "real_networks_datasets"
DATA_DIR = BASE_DIR / "outputs" / "data"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
for folder in (DATA_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)


%matplotlib inline
%config InlineBackend.figure_format = 'retina'

INVENTORY_CSV = DATA_DIR / 'real_network_inventory.csv'
METRICS_CSV = DATA_DIR / 'real_network_metrics.csv'
RANGES_CSV = DATA_DIR / 'real_network_plausibility_ranges.csv'
FIGURE_STEM = FIGURE_DIR / 'Fig_real_network_metrics'
OR_FIGURE_STEM = FIGURE_DIR / 'Fig_real_network_metrics_OR_sensitivity'
SAMPLE_FIGURE_STEM = FIGURE_DIR / 'Fig_real_network_benchmark_sensitivity'
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', context='notebook')

## 1 · Load and classify source networks

The collection contains 74 networks from 14 datasets. The benchmark flags separate comparable friendship-like relations from broader workplace and discussion relations.

The High School 2013 friendship file is a directed survey edge list. It contains 134 IDs that occur in at least one nomination; because no respondent roster accompanies the edge list here, a completely isolated respondent cannot be recovered and its roster is marked `edge_induced`.

The newly supplied Hi-tech and Prison files are directed friendship nominations with complete 36- and 67-node rosters. Their OR projections contain 91 and 142 edges, exactly matching Chang et al. (2025, Table C3). The added Karate and Tailor files are exact alternate copies of networks already loaded and are not double-counted. `kapmine.xml` is not part of Chang et al.'s eight-network reference set.


In [ ]:
def numeric_rows(path, expected_cols=None):
    """Read whitespace-separated numeric rows, ignoring legacy control/footer lines."""
    rows = []
    text = Path(path).read_text(encoding='utf-8', errors='ignore')
    for line in text.splitlines():
        tokens = line.split()
        if not tokens:
            continue
        try:
            row = [float(token) for token in tokens]
        except ValueError:
            continue
        if expected_cols is None or len(row) == expected_cols:
            rows.append(row)
    return rows



def read_repository_edge_list(path, directed):
    """Read comma- or whitespace-separated Network Repository edge lists."""
    edges = []
    text = Path(path).read_text(encoding='utf-8', errors='ignore')
    for line in text.splitlines():
        if not line.strip() or line.lstrip().startswith('%'):
            continue
        values = re.split(r'[,\s]+', line.strip())
        if len(values) >= 2:
            edges.append((int(values[0]), int(values[1])))
    graph_type = nx.DiGraph if directed else nx.Graph
    G = graph_type()
    if edges:
        max_node = max(max(u, v) for u, v in edges)
        G.add_nodes_from(range(1, max_node + 1))
        G.add_edges_from(edges)
    return G


def read_casos_xml(path):
    """Read one directed agent network from a CASOS DynamicNetwork XML file."""
    root = ET.parse(path).getroot()
    nodes = [
        node.attrib['id']
        for node in root.findall(".//nodeclass[@type='agent']/node")
    ]
    networks = root.findall('.//networks/network')
    if len(networks) != 1:
        raise ValueError(f'{path}: expected one network, found {len(networks)}')
    network = networks[0]
    if network.attrib.get('isDirected', '').lower() != 'true':
        raise ValueError(f'{path}: expected a directed nomination network')

    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for link in network.findall('link'):
        if float(link.attrib.get('value', 1)) != 0:
            source, target = link.attrib['source'], link.attrib['target']
            if source not in G or target not in G:
                raise ValueError(f'{path}: link endpoint absent from node roster')
            G.add_edge(source, target)
    return G

def read_square_matrix(path):
    """Read a square matrix while tolerating DOS EOF/control lines."""
    rows = numeric_rows(path)
    widths = Counter(len(row) for row in rows)
    if not widths:
        raise ValueError(f'No numeric rows found in {path}')
    width = max(widths, key=lambda value: (widths[value], value))
    matrix_rows = [row for row in rows if len(row) == width]
    if len(matrix_rows) != width:
        raise ValueError(f'{path}: expected a square matrix, found {len(matrix_rows)} x {width}')
    return np.asarray(matrix_rows, dtype=float)


def graph_from_matrix(matrix, missing_codes=(9, 10)):
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f'Adjacency matrix must be square, got {matrix.shape}')
    adjacency = np.asarray(matrix == 1, dtype=np.uint8)
    for code in missing_codes:
        adjacency[matrix == code] = 0
    np.fill_diagonal(adjacency, 0)
    return nx.from_numpy_array(adjacency, create_using=nx.DiGraph)


def set_metadata(
    G, dataset, relation, wave=None, subset='all', source_file=None,
    roster_status='complete', chang_reference=False,
):
    selfloops = list(nx.selfloop_edges(G))
    G.remove_edges_from(selfloops)
    G.graph.update({
        'dataset': dataset,
        'relation': relation,
        'wave': wave,
        'subset': subset,
        'source_file': str(source_file) if source_file else None,
        'roster_status': roster_status,
        'selfloops_removed': len(selfloops),
        'chang_reference': chang_reference,
    })
    parts = [dataset, relation]
    if wave is not None:
        parts.append(str(wave))
    if subset != 'all':
        parts.append(subset)
    G.graph['network'] = ' | '.join(parts)
    return G


def read_facebook_ego_networks(data_dir):
    """Reconstruct each SNAP Facebook ego network, including its ego-to-friend ties."""
    output = []
    edge_paths = sorted(data_dir.glob('*.edges'), key=lambda item: int(item.stem))
    for edge_path in edge_paths:
        ego = int(edge_path.stem)
        G = nx.Graph()
        G.add_edges_from(
            (int(source), int(target))
            for source, target in numeric_rows(edge_path, expected_cols=2)
        )

        # SNAP omits the focal user from .edges. Every ID in .feat is a
        # friend of the ego, so these ties complete the ego network.
        feat_path = edge_path.with_suffix('.feat')
        friend_ids = [
            int(line.split()[0])
            for line in feat_path.read_text(encoding='utf-8').splitlines()
            if line.strip()
        ]
        G.add_node(ego)
        G.add_edges_from((ego, friend) for friend in friend_ids)
        output.append(set_metadata(
            G, 'Facebook', 'friendship', subset=f'ego_{ego}', source_file=edge_path,
        ))
    return output


def read_sciences_po(path):
    """Rebuild directed nominations from the public ordered-dyad Stata file."""
    columns = ['uid_i1', 'uid_i2', 'LUL', 'LDL', 'LDLreverse', 'LAL']
    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore', message='One or more strings in the dta file',
            category=UnicodeWarning,
        )
        dyads = pd.read_stata(path, columns=columns, convert_categoricals=False)
    nodes = set(dyads['uid_i1'].astype(int)) | set(dyads['uid_i2'].astype(int))

    # The file contains both ordered versions of every dyad. On uid_i1 <
    # uid_i2 rows, LDLreverse is i1 -> i2 and LDL is i2 -> i1.
    dyads = dyads.loc[dyads['uid_i1'] < dyads['uid_i2']].copy()
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for row in dyads.itertuples(index=False):
        low, high = int(row.uid_i1), int(row.uid_i2)
        if row.LDLreverse == 1:
            G.add_edge(low, high)
        if row.LDL == 1:
            G.add_edge(high, low)

    # Check the reconstruction against the published OR (LUL) and AND (LAL) flags.
    or_count = nx.Graph(G).number_of_edges()
    mutual_count = sum(
        1 for u, v in G.edges() if u < v and G.has_edge(v, u)
    )
    if or_count != int(dyads['LUL'].sum()) or mutual_count != int(dyads['LAL'].sum()):
        raise ValueError('Sciences Po directed-network reconstruction failed validation')
    return G


def read_galesburg(path):
    """Parse the friendship and discussion networks from the Pajek file."""
    lines = Path(path).read_text(encoding='utf-8', errors='ignore').splitlines()
    output = []
    starts = [i for i, line in enumerate(lines) if line.lower().startswith('*network')]
    for start in starts:
        end = next(
            (i for i in range(start + 1, len(lines))
             if lines[i].lower().startswith(('*network', '*partition'))),
            len(lines),
        )
        name = lines[start].split(maxsplit=1)[1]
        vertices_idx = next(
            i for i in range(start, end) if lines[i].lower().startswith('*vertices')
        )
        n = int(lines[vertices_idx].split()[1])
        id_map = {}
        labels = []
        for line in lines[vertices_idx + 1:vertices_idx + 1 + n]:
            match = re.match(r'\s*(\d+)\s+"([^"]+)"', line)
            if not match:
                raise ValueError(f'Could not parse Pajek vertex line: {line}')
            internal, label = int(match.group(1)), match.group(2)
            id_map[internal] = label
            labels.append(label)

        G = nx.DiGraph()
        G.add_nodes_from(labels)
        arcs_idx = next(
            i for i in range(vertices_idx + n + 1, end)
            if lines[i].lower().startswith('*arcs')
        )
        for line in lines[arcs_idx + 1:end]:
            if line.startswith('*'):
                break
            tokens = line.split()
            if len(tokens) >= 2:
                G.add_edge(id_map[int(tokens[0])], id_map[int(tokens[1])])
        relation = 'friendship' if 'friends' in name.lower() else 'discussion'
        output.append(set_metadata(
            G, 'Galesburg', relation, source_file=path,
            chang_reference=relation == 'friendship',
        ))
    return output

In [ ]:
def load_real_networks(data_dir=REAL_NETWORK_DIR):
    graphs = []

    # High School 2013 friendship nominations.
    path = data_dir / 'HighSchool2013_friendship_network.csv'
    G = nx.DiGraph()
    G.add_edges_from((int(a), int(b)) for a, b in numeric_rows(path, expected_cols=2))
    graphs.append(set_metadata(
        G, 'HighSchool2013', 'friendship', source_file=path,
        roster_status='edge_induced',
    ))

    # Galesburg friendship and discussion networks.
    graphs.extend(read_galesburg(data_dir / 'Galesburg2.paj'))

    # Moreno High School: directed positive ties; weights are binarized.
    path = data_dir / 'soc-highschool-moreno' / 'soc-highschool-moreno.edges'
    G = read_repository_edge_list(path, directed=True)
    graphs.append(set_metadata(
        G, 'MorenoHighSchool', 'friendship', source_file=path,
        chang_reference=True,
    ))

    # Zachary karate club: intrinsically undirected.
    path = data_dir / 'ucidata-zachary' / 'ucidata-zachary.edges'
    G = read_repository_edge_list(path, directed=False)
    graphs.append(set_metadata(
        G, 'ZacharyKarate', 'social_ties', source_file=path,
        chang_reference=True,
    ))

    # Chang et al. reference networks: directed friendship nominations.
    # Their published edge counts correspond to the OR projections.
    for filename, dataset, expected_nodes, expected_or_edges in [
        ('hiTech/Hi-tech.xml', 'HiTech', 36, 91),
        ('prison.xml', 'Prison', 67, 142),
    ]:
        path = data_dir / filename
        G = read_casos_xml(path)
        or_edges = G.to_undirected().number_of_edges()
        if G.number_of_nodes() != expected_nodes or or_edges != expected_or_edges:
            raise ValueError(
                f'{path}: expected {expected_nodes} nodes and '
                f'{expected_or_edges} OR edges, found '
                f'{G.number_of_nodes()} and {or_edges}'
            )
        graphs.append(set_metadata(
            G, dataset, 'friendship', source_file=path, chang_reference=True,
        ))

    # Knecht class 12b: four longitudinal friendship waves.
    klas_dir = data_dir / 'klas12b'
    for wave in range(1, 5):
        path = klas_dir / f'klas12b-net-{wave}.dat'
        G = graph_from_matrix(read_square_matrix(path))
        graphs.append(set_metadata(G, 'Knecht12b', 'friendship', wave=wave, source_file=path))
    # Lazega lawyers: full-firm and partner-only advice, friendship, and cowork ties.
    lazega_dir = data_dir / 'LazegaLawyers'
    for filename, relation in [
        ('ELadv.dat', 'advice'), ('ELfriend.dat', 'friendship'), ('ELwork.dat', 'cowork'),
    ]:
        path = lazega_dir / filename
        G = graph_from_matrix(read_square_matrix(path))
        graphs.append(set_metadata(G, 'LazegaLawyers', relation, source_file=path))
    for filename, relation in [
        ('eladv36.dat', 'advice'), ('elfriend36.dat', 'friendship'), ('ELwork36.dat', 'cowork'),
    ]:
        path = lazega_dir / filename
        G = graph_from_matrix(read_square_matrix(path))
        graphs.append(set_metadata(G, 'LazegaLawyers', relation, subset='partners', source_file=path))

    # Kapferer tailor shop: instrumental and sociational ties at two times.
    kapf_dir = data_dir / 'kapfdata'
    for filename, relation, wave in [
        ('kapfti1.dat', 'instrumental', 1), ('kapfti2.dat', 'instrumental', 2),
        ('kapfts1.dat', 'sociational', 1), ('kapfts2.dat', 'sociational', 2),
    ]:
        path = kapf_dir / filename
        G = graph_from_matrix(read_square_matrix(path))
        graphs.append(set_metadata(
            G, 'Kapferer', relation, wave=wave, source_file=path,
            chang_reference=relation == 'sociational',
        ))

    # Baerveldt: 19 schools observed at two waves.
    cb_dir = data_dir / 'CB_data'
    school_ids = sorted(
        int(re.search(r'N34_(\d+)\.DAT$', item.name).group(1))
        for item in cb_dir.glob('N34_*.DAT')
    )
    for school in school_ids:
        for wave, filename in [(1, f'N34_{school}.DAT'), (2, f'HN34_{school}.DAT')]:
            path = cb_dir / filename
            G = graph_from_matrix(read_square_matrix(path))
            graphs.append(set_metadata(
                G, 'Baerveldt', f'emotional_support_school_{school}',
                wave=wave, source_file=path,
            ))

    # PEAS adolescent friendship nominations. The compact CSV is derived from
    # PEAS_edgelist_nomissing.sav (ID -> FRIEND); all 213 surveyed nodes are kept.
    path = data_dir / '4tgxy-osfstorage-archive' / 'PEAS_friendship_edges.csv'
    peas_edges = pd.read_csv(path, usecols=['source', 'target'])
    G = nx.DiGraph()
    G.add_nodes_from(range(1, 214))
    G.add_edges_from(peas_edges.itertuples(index=False, name=None))
    graphs.append(set_metadata(G, 'PEAS', 'friendship', source_file=path))

    # Sciences Po friendship nominations reconstructed from the public dyadic file.
    path = data_dir / 'dataverse_files' / 'Data' / 'Analysis' / 'work_data_public-1.dta'
    G = read_sciences_po(path)
    graphs.append(set_metadata(G, 'SciencesPo', 'friendship', source_file=path))

    # MSW student friendship nominations at waves 2--4. Retain the full
    # 145-student roster at each wave so nonparticipants and isolates are explicit.
    msw_dir = data_dir / 'UTA_Social_Networks_Social_Good'
    roster_path = msw_dir / 'ParticipationAcrossWaves.csv'
    roster = pd.read_csv(roster_path, usecols=['PARTID'])['PARTID'].astype(int)
    for wave in range(2, 5):
        path = msw_dir / f'w{wave}_FriendshipEdgelist.csv'
        edges = pd.read_csv(path, usecols=['PARTID', 'Alter']).dropna()
        edges = edges.astype({'PARTID': int, 'Alter': int})
        G = nx.DiGraph()
        G.add_nodes_from(roster)
        G.add_edges_from(edges.itertuples(index=False, name=None))
        graphs.append(set_metadata(
            G, 'MSWStudents', 'friendship', wave=wave, source_file=path,
        ))

    # SNAP Facebook ego networks. These are intrinsically undirected.
    graphs.extend(read_facebook_ego_networks(data_dir / 'facebook'))

    names = [G.graph['network'] for G in graphs]
    if len(names) != len(set(names)):
        duplicates = [name for name, count in Counter(names).items() if count > 1]
        raise ValueError(f'Duplicate network names: {duplicates}')
    return {G.graph['network']: G for G in graphs}


def relation_family(relation):
    if relation == 'friendship':
        return 'friendship'
    if (
        relation in {'social_ties', 'sociational'}
        or relation.startswith('emotional_support_school_')
    ):
        return 'friendship_like'
    return 'other_social_relation'


graphs = load_real_networks()
for G in graphs.values():
    family = relation_family(G.graph['relation'])
    is_facebook = G.graph['dataset'] == 'Facebook'
    G.graph.update({
        'relation_family': family,
        'primary_benchmark': family in {'friendship', 'friendship_like'} and not is_facebook,
        'strict_friendship': family == 'friendship' and not is_facebook,
        'facebook_sensitivity': family in {'friendship', 'friendship_like'},
    })

inventory_df = pd.DataFrame([
    {
        'network': name,
        'dataset': G.graph['dataset'],
        'relation': G.graph['relation'],
        'relation_family': G.graph['relation_family'],
        'wave': G.graph['wave'],
        'subset': G.graph['subset'],
        'nodes': G.number_of_nodes(),
        'edges_or_arcs': G.number_of_edges(),
        'source_directed': G.is_directed(),
        'roster_status': G.graph['roster_status'],
        'selfloops_removed': G.graph['selfloops_removed'],
        'primary_benchmark': G.graph['primary_benchmark'],
        'strict_friendship': G.graph['strict_friendship'],
        'facebook_sensitivity': G.graph['facebook_sensitivity'],
        'chang_reference': G.graph['chang_reference'],
        'source_file': G.graph['source_file'],
    }
    for name, G in graphs.items()
]).sort_values(['dataset', 'relation', 'wave'], na_position='last').reset_index(drop=True)

expected_by_dataset = {
    'HighSchool2013': 1, 'Galesburg': 2, 'MorenoHighSchool': 1,
    'ZacharyKarate': 1, 'Knecht12b': 4, 'LazegaLawyers': 6,
    'Kapferer': 4, 'Baerveldt': 38, 'PEAS': 1, 'SciencesPo': 1,
    'MSWStudents': 3, 'Facebook': 10, 'HiTech': 1, 'Prison': 1,
}
observed_by_dataset = inventory_df.groupby('dataset').size().to_dict()
if observed_by_dataset != expected_by_dataset:
    raise ValueError(f'Unexpected dataset inventory: {observed_by_dataset}')
if any(nx.number_of_selfloops(G) for G in graphs.values()):
    raise ValueError('At least one source graph contains a self-loop.')
if int(inventory_df['selfloops_removed'].sum()) != 12:
    raise ValueError('Expected to remove 12 raw self-nominations.')
if int(inventory_df['primary_benchmark'].sum()) != 57:
    raise ValueError('Primary benchmark must contain 57 networks.')
if int(inventory_df['strict_friendship'].sum()) != 16:
    raise ValueError('Strict-friendship sensitivity must contain 16 networks.')
if int(inventory_df['facebook_sensitivity'].sum()) != 67:
    raise ValueError('Facebook sensitivity must contain 67 networks.')
if int(inventory_df['chang_reference'].sum()) != 7:
    raise ValueError("Expected seven available networks from Chang et al.'s eight.")

inventory_df.to_csv(INVENTORY_CSV, index=False)
print(
    f'Validated {len(graphs)} networks from {inventory_df.dataset.nunique()} datasets; '
    f"primary={inventory_df['primary_benchmark'].sum()}, "
    f"strict={inventory_df['strict_friendship'].sum()}, "
    f"with Facebook={inventory_df['facebook_sensitivity'].sum()}, "
    f"Chang subset={inventory_df['chang_reference'].sum()}."
)
display(inventory_df)


## 2 · Undirected Projection and Structural Metric Functions

In [ ]:
def undirected_projection(G, method):
    """Build a Mutual or OR projection while retaining every source node."""
    if method not in {'Mutual', 'OR'}:
        raise ValueError("method must be 'Mutual' or 'OR'")

    if not G.is_directed():
        U = nx.Graph(G)
    else:
        U = nx.Graph()
        U.add_nodes_from(G.nodes(data=True))
        if method == 'Mutual':
            U.add_edges_from(
                (u, v) for u, v in G.edges()
                if u != v and G.has_edge(v, u)
            )
        else:
            U.add_edges_from((u, v) for u, v in G.edges() if u != v)

    U.remove_edges_from(nx.selfloop_edges(U))
    U.graph.update(G.graph)
    U.graph['projection'] = method
    return U


def gini(values):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0 or arr.sum() == 0:
        return 0.0
    arr = np.sort(arr)
    n = len(arr)
    idx = np.arange(1, n + 1)
    return (2 * (idx * arr).sum()) / (n * arr.sum()) - (n + 1) / n


def degree_stats(G):
    # Matches the mutual-network notebook: average of NetworkX degree.
    degree = [value for _, value in G.degree()]
    return {
        'avg_degree': np.mean(degree),
        'degree_std': np.std(degree),
        'degree_max': np.max(degree),
        'degree_gini': gini(degree),
    }


def clustering_stats(G):
    values = list(nx.clustering(G).values())
    return {
        'clustering_mean': np.mean(values),
        'clustering_std': np.std(values),
        'clustering_zero_frac': np.mean(np.asarray(values) == 0),
    }


def component_stats(G):
    n = G.number_of_nodes()
    if n == 0:
        return {
            'lcc_n': 0, 'lcc_ratio': np.nan, 'avg_path_length': np.nan,
            'normalized_path_length': np.nan,
        }
    largest = max(nx.connected_components(G), key=len)
    lcc_n = len(largest)
    path_length = nx.average_shortest_path_length(G.subgraph(largest))
    normalized_path = path_length / np.log(lcc_n) if lcc_n > 1 else np.nan
    return {
        'lcc_n': lcc_n,
        'lcc_ratio': lcc_n / n,
        'avg_path_length': path_length,
        'normalized_path_length': normalized_path,
    }


def louvain_modularity(G):
    if G.number_of_edges() == 0:
        return np.nan
    communities = louvain_communities(G, seed=RANDOM_SEED)
    return modularity(G, communities)


def compute_structural_metrics(G):
    row = {
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges(),
        'density': nx.density(G),
    }
    row.update(degree_stats(G))
    row.update(clustering_stats(G))
    row.update(component_stats(G))
    row['louvain_modularity'] = louvain_modularity(G)
    return row

## 3 · Compute Mutual and OR Network Metrics

In [ ]:
records = []
projected_graphs = {}

for name, source_G in graphs.items():
    for method in ('Mutual', 'OR'):
        G = undirected_projection(source_G, method)
        projected_graphs[(name, method)] = G
        row = {
            'network': name,
            'projection': method,
            'dataset': source_G.graph['dataset'],
            'relation': source_G.graph['relation'],
            'relation_family': source_G.graph['relation_family'],
            'wave': source_G.graph['wave'],
            'subset': source_G.graph['subset'],
            'source_directed': source_G.is_directed(),
            'source_file': source_G.graph['source_file'],
            'roster_status': source_G.graph['roster_status'],
            'primary_benchmark': source_G.graph['primary_benchmark'],
            'strict_friendship': source_G.graph['strict_friendship'],
            'facebook_sensitivity': source_G.graph['facebook_sensitivity'],
            'chang_reference': source_G.graph['chang_reference'],
        }
        row.update(compute_structural_metrics(G))
        records.append(row)

metrics_df = pd.DataFrame(records).sort_values(
    ['projection', 'dataset', 'relation', 'wave', 'subset'], na_position='last'
).reset_index(drop=True)
metrics_df.to_csv(METRICS_CSV, index=False)

print(
    f'Saved {len(metrics_df)} metric rows ({len(graphs)} source networks × 2 projections) '
    f'-> {METRICS_CSV}'
)
if len(metrics_df) != 148 or metrics_df[['network', 'projection']].duplicated().any():
    raise ValueError('Expected 74 unique networks in each of two projections.')
if not (
    metrics_df.pivot(index='network', columns='projection', values='n_edges')['Mutual']
    <= metrics_df.pivot(index='network', columns='projection', values='n_edges')['OR']
).all():
    raise ValueError('A Mutual projection has more edges than its OR projection.')

chang_expected_or_edges = {
    'Galesburg | friendship': 63,
    'HiTech | friendship': 91,
    'ZacharyKarate | social_ties': 78,
    'Prison | friendship': 142,
    'Kapferer | sociational | 1': 158,
    'Kapferer | sociational | 2': 223,
    'MorenoHighSchool | friendship': 274,
}
chang_or = (
    metrics_df[metrics_df['projection'].eq('OR') & metrics_df['chang_reference']]
    .set_index('network')['n_edges'].to_dict()
)
if chang_or != chang_expected_or_edges:
    raise ValueError(f'Chang-reference validation failed: {chang_or}')
display(metrics_df.round(4))

## 4 · Descriptive summaries

In [ ]:
PLAUSIBILITY_METRICS = [
    'avg_degree', 'clustering_mean', 'lcc_ratio',
    'normalized_path_length', 'louvain_modularity',
]
DESCRIPTIVE_METRICS = ['density', 'avg_path_length'] + PLAUSIBILITY_METRICS

dataset_summary = (
    metrics_df
    .groupby(['projection', 'dataset'])[DESCRIPTIVE_METRICS]
    .agg(['mean', 'std', 'count'])
    .round(4)
)
display(dataset_summary)


## 5 · Empirical plausibility ranges

For the primary ranges, every dataset receives equal total weight and networks within a dataset divide that weight equally. A value is called empirically plausible when it lies in the weighted central 95% interval. Equal-network ranges and broader samples are retained as sensitivity analyses.


In [ ]:
def weighted_quantile(values, weights, probabilities=(.025, .5, .975)):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[valid], weights[valid]
    if values.size == 0:
        return np.full(len(probabilities), np.nan)
    order = np.argsort(values, kind='mergesort')
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights) / weights.sum()
    indices = np.searchsorted(cumulative, probabilities, side='left')
    return values[np.clip(indices, 0, len(values) - 1)]


BENCHMARK_SAMPLES = {
    'primary_friendship_like': lambda frame: frame['primary_benchmark'],
    'strict_friendship': lambda frame: frame['strict_friendship'],
    'chang_available': lambda frame: frame['chang_reference'],
    'friendship_like_with_facebook': lambda frame: frame['facebook_sensitivity'],
    'all_relations': lambda frame: pd.Series(True, index=frame.index),
}


def compute_plausibility_ranges(data):
    rows = []
    for sample, selector in BENCHMARK_SAMPLES.items():
        sample_data = data.loc[selector(data)].copy()
        for projection, projected in sample_data.groupby('projection', observed=True):
            for weighting in ('equal_dataset', 'equal_network'):
                if weighting == 'equal_dataset':
                    within_dataset_n = projected.groupby('dataset')['network'].transform('count')
                    weights = 1 / (projected['dataset'].nunique() * within_dataset_n)
                else:
                    weights = pd.Series(1 / len(projected), index=projected.index)
                for metric in PLAUSIBILITY_METRICS:
                    lower, median, upper = weighted_quantile(projected[metric], weights)
                    rows.append({
                        'sample': sample,
                        'projection': projection,
                        'weighting': weighting,
                        'metric': metric,
                        'lower_2_5': lower,
                        'median': median,
                        'upper_97_5': upper,
                        'n_networks': projected['network'].nunique(),
                        'n_datasets': projected['dataset'].nunique(),
                    })
    return pd.DataFrame(rows)


plausibility_ranges_df = compute_plausibility_ranges(metrics_df)
plausibility_ranges_df.to_csv(RANGES_CSV, index=False)

primary_ranges = plausibility_ranges_df[
    (plausibility_ranges_df['sample'] == 'primary_friendship_like')
    & (plausibility_ranges_df['projection'] == 'Mutual')
    & (plausibility_ranges_df['weighting'] == 'equal_dataset')
]
if len(primary_ranges) != len(PLAUSIBILITY_METRICS):
    raise ValueError('The primary benchmark must define all five metric ranges.')
if not (primary_ranges['n_networks'].eq(57) & primary_ranges['n_datasets'].eq(13)).all():
    raise ValueError('The primary ranges must use 57 networks from 13 datasets.')

print(f'Saved {len(plausibility_ranges_df)} primary/sensitivity range rows -> {RANGES_CSV}')
display(primary_ranges.round(4))


## 6 · Structural benchmark figures

The main figure shows the 57-network primary sample under the Mutual projection. The shaded region is the equal-study weighted central 95% interval and the dashed line is its weighted median. OR projection and sample-definition comparisons are saved as sensitivity figures, including the seven locally available networks from Chang et al. (2025).


In [ ]:
DATASET_ORDER = [
    'HighSchool2013', 'MorenoHighSchool', 'ZacharyKarate', 'Galesburg',
    'HiTech', 'Prison',
    'Knecht12b', 'LazegaLawyers', 'Kapferer', 'Baerveldt',
    'PEAS', 'SciencesPo', 'MSWStudents', 'Facebook',
]

DATASET_LABELS = {
    'HighSchool2013': 'High School 2013',
    'MorenoHighSchool': 'Moreno High School',
    'ZacharyKarate': 'Zachary Karate',
    'Galesburg': 'Galesburg',
    'HiTech': 'Hi-tech',
    'Prison': 'Prison',
    'Knecht12b': 'Knecht 12b',
    'LazegaLawyers': 'Lazega Lawyers',
    'Kapferer': 'Kapferer',
    'Baerveldt': 'Baerveldt',
    'PEAS': 'PEAS',
    'SciencesPo': 'Sciences Po',
    'MSWStudents': 'MSW Students',
    'Facebook': 'Facebook',
}

DATASET_COLORS = dict(zip(
    DATASET_ORDER,
    sns.color_palette('husl', len(DATASET_ORDER)).as_hex(),
))

PROJECTION_ORDER = ['Mutual', 'OR']
PROJECTION_LABELS = {'Mutual': 'Mutual', 'OR': 'OR (either direction)'}

METRIC_LABELS = {
    'avg_degree': 'Average degree',
    'clustering_mean': 'Average clustering',
    'lcc_ratio': 'Nodes in LCC',
    'normalized_path_length': r'Normalized path $\ell/\log(N_{LCC})$',
    'louvain_modularity': 'Modularity',
}


def dataset_metric_summary(data):
    long = data.melt(
        id_vars=['projection', 'dataset'],
        value_vars=PLAUSIBILITY_METRICS,
        var_name='metric', value_name='value',
    )
    return (
        long.groupby(['projection', 'dataset', 'metric'], observed=True)['value']
        .agg(mean='mean', std='std', n='count').reset_index()
    )


def plot_empirical_metrics(data, projection, output_stem, title):
    selected = data[
        data['primary_benchmark'] & data['projection'].eq(projection)
    ].copy()
    datasets = [dataset for dataset in DATASET_ORDER if dataset in set(selected['dataset'])]
    ranges = plausibility_ranges_df[
        plausibility_ranges_df['sample'].eq('primary_friendship_like')
        & plausibility_ranges_df['projection'].eq(projection)
        & plausibility_ranges_df['weighting'].eq('equal_dataset')
    ].set_index('metric')
    rng = np.random.default_rng(RANDOM_SEED)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4.8), constrained_layout=False)
    x = np.arange(len(datasets))

    for ax, metric in zip(axes, PLAUSIBILITY_METRICS):
        interval = ranges.loc[metric]
        ax.axhspan(interval['lower_2_5'], interval['upper_97_5'], color='#D9D9D9', alpha=.45)
        ax.axhline(interval['median'], color='#555555', linestyle='--', linewidth=1.1)
        for index, dataset in enumerate(datasets):
            values = selected.loc[selected['dataset'].eq(dataset), metric].dropna().to_numpy()
            jitter = rng.uniform(-.09, .09, len(values))
            color = DATASET_COLORS[dataset]
            ax.scatter(np.full(len(values), index) + jitter, values, s=20, color=color, alpha=.35, linewidth=0)
            ax.scatter(index, values.mean(), s=58, color=color, edgecolor='white', linewidth=.8, zorder=3)
        ax.set_title(METRIC_LABELS[metric], fontsize=14)
        ax.set_xticks(x, [DATASET_LABELS[d] for d in datasets], rotation=65, ha='right')
        ax.tick_params(axis='x', labelsize=7)
        ax.grid(axis='y', color='#E1E1E1', linewidth=.7)
        ax.spines[['top', 'right']].set_visible(False)
        if metric in {'clustering_mean', 'lcc_ratio', 'louvain_modularity'}:
            ax.set_ylim(-.02, 1.02)
    axes[0].set_ylabel('Metric value', fontsize=12)
    fig.suptitle(title, fontsize=17, y=.99)
    handles = [
        Line2D([0], [0], marker='o', linestyle='', color='#777777', alpha=.4, label='Network / wave / layer'),
        Line2D([0], [0], marker='o', linestyle='', color='#777777', markersize=8, label='Dataset mean'),
        Line2D([0], [0], color='#777777', linewidth=8, alpha=.25, label='Weighted central 95%'),
        Line2D([0], [0], color='#555555', linestyle='--', label='Weighted median'),
    ]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(.5, .01), ncol=4, frameon=False)
    fig.subplots_adjust(left=.055, right=.995, top=.84, bottom=.34, wspace=.28)
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


def plot_sample_sensitivity(ranges, output_stem):
    labels = {
        'primary_friendship_like': 'Primary friendship-like',
        'strict_friendship': 'Strict friendship',
        'chang_available': 'Chang subset (7 of 8)',
        'friendship_like_with_facebook': 'Primary + Facebook',
        'all_relations': 'All relations',
    }
    order = list(labels)
    selected = ranges[
        ranges['projection'].eq('Mutual') & ranges['weighting'].eq('equal_dataset')
    ]
    fig, axes = plt.subplots(1, 5, figsize=(18.5, 4.2))
    for ax, metric in zip(axes, PLAUSIBILITY_METRICS):
        metric_data = selected[selected['metric'].eq(metric)].set_index('sample').loc[order]
        y = np.arange(len(order))
        ax.errorbar(
            metric_data['median'], y,
            xerr=[metric_data['median'] - metric_data['lower_2_5'], metric_data['upper_97_5'] - metric_data['median']],
            fmt='o', color='#4C78A8', capsize=3, linewidth=1.4,
        )
        ax.set_title(METRIC_LABELS[metric], fontsize=13)
        ax.set_yticks(y, [labels[name] for name in order] if ax is axes[0] else [])
        ax.invert_yaxis()
        ax.grid(axis='x', color='#E1E1E1', linewidth=.7)
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle('Benchmark-definition sensitivity (Mutual projection)', fontsize=16)
    fig.subplots_adjust(left=.13, right=.995, top=.82, bottom=.16, wspace=.32)
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


real_network_metrics_summary = dataset_metric_summary(metrics_df)
real_network_metrics_summary.to_csv(f'{FIGURE_STEM}_summary.csv', index=False)
fig_real_network_metrics = plot_empirical_metrics(
    metrics_df, 'Mutual', FIGURE_STEM,
    'Primary empirical benchmark: Mutual friendship-like networks',
)
fig_real_network_metrics_or = plot_empirical_metrics(
    metrics_df, 'OR', OR_FIGURE_STEM,
    'Projection sensitivity: OR friendship-like networks',
)
fig_real_network_sample_sensitivity = plot_sample_sensitivity(
    plausibility_ranges_df, SAMPLE_FIGURE_STEM,
)

print('Saved primary Mutual, OR-projection, and benchmark-definition figures.')


## 7 · Normalized degree distributions

Node degree is normalized as $d_i/(N-1)$. The underlying CSV keeps a shared 50-bin grid for comparison with notebook 02, while displayed histograms combine adjacent bins into 25 wider bins. Curves remain available for all datasets; the pooled reference uses only the 57-network primary benchmark and gives equal weight first to networks within a study and then to the 13 studies. Mutual and OR projections are kept separate.


In [ ]:
REAL_DEGREE_STEM = FIGURE_DIR / 'Fig_normalized_degree_distribution_real_networks'
REAL_DEGREE_HIST_STEM = FIGURE_DIR / 'Fig_normalized_degree_histogram_real_networks'
REAL_DEGREE_COMBINED_STEM = FIGURE_DIR / 'Fig_normalized_degree_distribution_real_networks_all_datasets'
REAL_DEGREE_POOLED_STEM = FIGURE_DIR / 'Fig_normalized_degree_histogram_real_networks_pooled'
REAL_DEGREE_BINS = np.linspace(0, 1, 51)
REAL_DEGREE_BIN_CENTERS = (REAL_DEGREE_BINS[:-1] + REAL_DEGREE_BINS[1:]) / 2
REAL_HISTOGRAM_BINS = np.linspace(0, 1, 26)


def coarsen_real_histogram(probability):
    probability = np.asarray(probability, dtype=float)
    if len(probability) != 50:
        raise ValueError('Expected the shared 50-bin degree grid.')
    return probability.reshape(25, 2).sum(axis=1)


def build_real_degree_distribution_data(projected):
    rows = []
    for (network, projection), G in projected.items():
        n = G.number_of_nodes()
        normalized_degree = np.asarray(
            [degree / (n - 1) if n > 1 else 0 for _, degree in G.degree()],
            dtype=float,
        )
        counts, _ = np.histogram(normalized_degree, bins=REAL_DEGREE_BINS)
        probability = counts / counts.sum()
        ccdf = probability[::-1].cumsum()[::-1]
        metadata = {
            'network': network,
            'dataset': G.graph['dataset'],
            'relation': G.graph['relation'],
            'wave': G.graph['wave'],
            'subset': G.graph['subset'],
            'projection': projection,
            'n_nodes': n,
            'relation_family': G.graph['relation_family'],
            'primary_benchmark': G.graph['primary_benchmark'],
            'strict_friendship': G.graph['strict_friendship'],
            'chang_reference': G.graph['chang_reference'],
            'facebook_sensitivity': G.graph['facebook_sensitivity'],
        }
        for bin_index, (left, right, center, count, prob, survival) in enumerate(zip(
            REAL_DEGREE_BINS[:-1], REAL_DEGREE_BINS[1:],
            REAL_DEGREE_BIN_CENTERS, counts, probability, ccdf,
        )):
            rows.append({
                **metadata,
                'bin_index': bin_index,
                'normalized_degree_left': left,
                'normalized_degree_right': right,
                'normalized_degree_center': center,
                'count': int(count),
                'probability': prob,
                'ccdf': survival,
            })
    return pd.DataFrame(rows)


def _real_degree_axes():
    fig, axes = plt.subplots(
        4, 4, figsize=(17.5, 15.5), sharex=True, sharey=True,
        constrained_layout=False,
    )
    return fig, axes.ravel()


def _style_real_degree_axis(ax, dataset, ylabel=None, log_y=False):
    ax.set_title(DATASET_LABELS[dataset].replace('\n', ' '), fontsize=14, pad=7)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, .25, .5, .75, 1])
    if log_y:
        ax.set_yscale('log')
        ax.set_ylim(1e-4, 1.08)
        ax.grid(True, which='major', color='#D9D9D9', linewidth=.7, alpha=.75)
        ax.grid(True, which='minor', axis='y', color='#ECECEC', linewidth=.4, alpha=.6)
    else:
        ax.set_ylim(bottom=0)
        ax.grid(True, axis='y', color='#D9D9D9', linewidth=.7, alpha=.75)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(axis='both', labelsize=10, length=3.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#777777')
    ax.spines[['left', 'bottom']].set_linewidth(.65)


def plot_real_degree_ccdfs(distributions, projection, output_stem):
    fig, axes = _real_degree_axes()
    projection_data = distributions[distributions['projection'] == projection]

    for panel, (ax, dataset) in enumerate(zip(axes, DATASET_ORDER)):
        subset = projection_data[projection_data['dataset'] == dataset]
        color = DATASET_COLORS[dataset]
        for _, curve in subset.groupby('network', observed=True):
            curve = curve.sort_values('bin_index')
            positive = curve['ccdf'] > 0
            ax.step(
                curve.loc[positive, 'normalized_degree_left'],
                curve.loc[positive, 'ccdf'],
                where='post', color=color, linewidth=.7, alpha=.20,
            )
        mean_curve = (
            subset.groupby('bin_index', observed=True)
            .agg(normalized_degree_left=('normalized_degree_left', 'first'),
                 ccdf=('ccdf', 'mean'))
            .reset_index().sort_values('bin_index')
        )
        positive = mean_curve['ccdf'] > 0
        ax.step(
            mean_curve.loc[positive, 'normalized_degree_left'],
            mean_curve.loc[positive, 'ccdf'],
            where='post', color=color, linewidth=2.4, alpha=.98,
        )
        _style_real_degree_axis(
            ax, dataset, ylabel=r'$P(d/(N-1) \geq x)$' if panel % 4 == 0 else None,
            log_y=True,
        )
    for ax in axes[len(DATASET_ORDER):]:
        ax.set_visible(False)

    fig.supxlabel('Normalized degree $x = d/(N-1)$', fontsize=15, y=.055)
    fig.suptitle(f'{PROJECTION_LABELS[projection]} projection', fontsize=18, y=.985)
    handles = [
        Line2D([0], [0], color='#777777', linewidth=.8, alpha=.3,
               label='Individual network / layer / wave'),
        Line2D([0], [0], color='#777777', linewidth=2.5,
               label='Equal-weight dataset mean'),
    ]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(.5, .005),
               ncol=2, frameon=False, fontsize=12)
    fig.subplots_adjust(left=.07, right=.99, top=.93, bottom=.11,
                        hspace=.30, wspace=.18)
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


def plot_real_degree_histograms(distributions, projection, output_stem):
    fig, axes = _real_degree_axes()
    projection_data = distributions[distributions['projection'] == projection]

    for panel, (ax, dataset) in enumerate(zip(axes, DATASET_ORDER)):
        subset = projection_data[projection_data['dataset'] == dataset]
        color = DATASET_COLORS[dataset]
        for _, histogram in subset.groupby('network', observed=True):
            histogram = histogram.sort_values('bin_index')
            ax.stairs(
                coarsen_real_histogram(histogram['probability']),
                REAL_HISTOGRAM_BINS, baseline=None,
                color=color, linewidth=.7, alpha=.20,
            )
        mean_probability = (
            subset.groupby('bin_index', observed=True)['probability']
            .mean().reindex(range(len(REAL_DEGREE_BIN_CENTERS)), fill_value=0)
        )
        ax.stairs(
            coarsen_real_histogram(mean_probability),
            REAL_HISTOGRAM_BINS, baseline=None,
            color=color, linewidth=2.4, alpha=.98,
        )
        _style_real_degree_axis(
            ax, dataset, ylabel='Probability per bin' if panel % 4 == 0 else None,
        )
    for ax in axes[len(DATASET_ORDER):]:
        ax.set_visible(False)

    fig.supxlabel('Normalized degree $d/(N-1)$', fontsize=15, y=.055)
    fig.suptitle(f'{PROJECTION_LABELS[projection]} projection', fontsize=18, y=.985)
    handles = [
        Line2D([0], [0], color='#777777', linewidth=.8, alpha=.3,
               label='Individual network / layer / wave'),
        Line2D([0], [0], color='#777777', linewidth=2.5,
               label='Equal-weight dataset mean'),
    ]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(.5, .005),
               ncol=2, frameon=False, fontsize=12)
    fig.subplots_adjust(left=.07, right=.99, top=.93, bottom=.11,
                        hspace=.30, wspace=.18)
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


def plot_all_real_degree_distributions(distributions, output_stem):
    fig, axes = plt.subplots(
        2, 2, figsize=(15.5, 10.5), sharex=True,
        gridspec_kw={'width_ratios': [1, 1]}, constrained_layout=False,
    )

    for row_i, projection in enumerate(PROJECTION_ORDER):
        projection_data = distributions[distributions['projection'] == projection]
        hist_ax, ccdf_ax = axes[row_i]
        for dataset in DATASET_ORDER:
            subset = projection_data[projection_data['dataset'] == dataset]
            if subset.empty:
                continue
            dataset_mean = (
                subset.groupby('bin_index', observed=True)
                .agg(
                    probability=('probability', 'mean'),
                    ccdf=('ccdf', 'mean'),
                    normalized_degree_left=('normalized_degree_left', 'first'),
                )
                .reset_index().sort_values('bin_index')
            )
            color = DATASET_COLORS[dataset]
            hist_ax.stairs(
                coarsen_real_histogram(dataset_mean['probability']),
                REAL_HISTOGRAM_BINS, baseline=None,
                color=color, linewidth=1.8, alpha=.92,
            )
            positive = dataset_mean['ccdf'] > 0
            ccdf_ax.step(
                dataset_mean.loc[positive, 'normalized_degree_left'],
                dataset_mean.loc[positive, 'ccdf'],
                where='post', color=color, linewidth=1.8, alpha=.92,
            )

        hist_ax.set_ylabel(
            f"{PROJECTION_LABELS[projection]}\nProbability per bin", fontsize=13,
        )
        hist_ax.set_ylim(bottom=0)
        ccdf_ax.set_ylabel(
            f"{PROJECTION_LABELS[projection]}\n$P(d/(N-1) \\geq x)$", fontsize=13,
        )
        ccdf_ax.set_yscale('log')
        ccdf_ax.set_ylim(1e-4, 1.08)
        for ax in (hist_ax, ccdf_ax):
            ax.set_xlim(0, 1)
            ax.set_xticks([0, .25, .5, .75, 1])
            ax.grid(True, which='major', color='#D9D9D9', linewidth=.7, alpha=.75)
            ax.tick_params(axis='both', labelsize=10, length=3.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.spines[['left', 'bottom']].set_color('#777777')
            ax.spines[['left', 'bottom']].set_linewidth(.65)
        ccdf_ax.grid(
            True, which='minor', axis='y', color='#ECECEC',
            linewidth=.4, alpha=.6,
        )

    axes[0, 0].set_title('Normalized-degree histogram', fontsize=16, pad=10)
    axes[0, 1].set_title('Normalized-degree CCDF', fontsize=16, pad=10)
    axes[1, 0].set_xlabel('Normalized degree $d/(N-1)$', fontsize=14)
    axes[1, 1].set_xlabel('Normalized degree $x = d/(N-1)$', fontsize=14)
    handles = [
        Line2D(
            [0], [0], color=DATASET_COLORS[dataset], linewidth=2.2,
            label=DATASET_LABELS[dataset].replace('\n', ' '),
        )
        for dataset in DATASET_ORDER
    ]
    fig.legend(
        handles=handles, loc='lower center', bbox_to_anchor=(.5, .005),
        ncol=4, frameon=False, fontsize=10.5, handlelength=2.4,
        columnspacing=1.4,
    )
    fig.subplots_adjust(
        left=.10, right=.985, top=.93, bottom=.16, hspace=.27, wspace=.22,
    )
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


def plot_pooled_real_degree_histograms(distributions, output_stem):
    # The pooled reference is the primary benchmark only. First average
    # networks within each dataset, then average datasets.
    benchmark_data = distributions[distributions['primary_benchmark']].copy()
    dataset_level = (
        benchmark_data
        .groupby(['projection', 'dataset', 'bin_index'], observed=True)
        .agg(
            probability=('probability', 'mean'),
            normalized_degree_left=('normalized_degree_left', 'first'),
            normalized_degree_right=('normalized_degree_right', 'first'),
            normalized_degree_center=('normalized_degree_center', 'first'),
            n_networks=('network', 'nunique'),
        )
        .reset_index()
    )
    pooled = (
        dataset_level
        .groupby(['projection', 'bin_index'], observed=True)
        .agg(
            probability=('probability', 'mean'),
            normalized_degree_left=('normalized_degree_left', 'first'),
            normalized_degree_right=('normalized_degree_right', 'first'),
            normalized_degree_center=('normalized_degree_center', 'first'),
            n_datasets=('dataset', 'nunique'),
            n_networks=('n_networks', 'sum'),
        )
        .reset_index().sort_values(['projection', 'bin_index'])
    )

    projection_colors = {'Mutual': '#4C78A8', 'OR': '#F58518'}
    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), sharex=True, sharey=True)
    for ax, projection in zip(axes, PROJECTION_ORDER):
        values = pooled[pooled['projection'] == projection].sort_values('bin_index')
        display_probability = coarsen_real_histogram(values['probability'])
        display_centers = (REAL_HISTOGRAM_BINS[:-1] + REAL_HISTOGRAM_BINS[1:]) / 2
        ax.bar(
            display_centers, display_probability,
            width=np.diff(REAL_HISTOGRAM_BINS), align='center',
            color=projection_colors[projection], edgecolor='white', linewidth=.35,
            alpha=.88,
        )
        ax.set_title(PROJECTION_LABELS[projection], fontsize=16, pad=9)
        ax.set_xlim(0, 1)
        ax.set_ylim(bottom=0)
        ax.set_xticks([0, .25, .5, .75, 1])
        ax.set_xlabel('Normalized degree $d/(N-1)$', fontsize=13)
        ax.grid(True, axis='y', color='#D9D9D9', linewidth=.7, alpha=.75)
        ax.tick_params(axis='both', labelsize=10, length=3.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_color('#777777')
        ax.spines[['left', 'bottom']].set_linewidth(.65)
    axes[0].set_ylabel('Probability per bin', fontsize=13)
    fig.suptitle(
        'Primary friendship-like benchmark (equal study weight)',
        fontsize=18, y=.99,
    )
    fig.subplots_adjust(left=.075, right=.99, top=.86, bottom=.14, wspace=.13)
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    pooled.to_csv(f'{output_stem}.csv', index=False)
    return fig, pooled


real_degree_distribution_df = build_real_degree_distribution_data(projected_graphs)
real_degree_distribution_df.to_csv(f'{REAL_DEGREE_STEM}.csv', index=False)

fig_real_degree_ccdf_mutual = plot_real_degree_ccdfs(
    real_degree_distribution_df, 'Mutual', f'{REAL_DEGREE_STEM}_Mutual',
)
fig_real_degree_ccdf_or = plot_real_degree_ccdfs(
    real_degree_distribution_df, 'OR', f'{REAL_DEGREE_STEM}_OR',
)
fig_real_degree_hist_mutual = plot_real_degree_histograms(
    real_degree_distribution_df, 'Mutual', f'{REAL_DEGREE_HIST_STEM}_Mutual',
)
fig_real_degree_hist_or = plot_real_degree_histograms(
    real_degree_distribution_df, 'OR', f'{REAL_DEGREE_HIST_STEM}_OR',
)
fig_real_degree_all_datasets = plot_all_real_degree_distributions(
    real_degree_distribution_df, REAL_DEGREE_COMBINED_STEM,
)
fig_real_degree_pooled, real_degree_pooled_df = plot_pooled_real_degree_histograms(
    real_degree_distribution_df, REAL_DEGREE_POOLED_STEM,
)

probability_checks = (
    real_degree_distribution_df
    .groupby(['network', 'projection'], observed=True)['probability'].sum()
)
print(
    f'Saved normalized-degree CSV, per-dataset figures, combined curves, and pooled histograms for '
    f'both projections; {len(graphs)} networks, '
    f'{len(real_degree_distribution_df):,} network-bin rows. '
    f'Probability sums: {probability_checks.min():.6f}--{probability_checks.max():.6f}.'
)

## Output files

- `real_network_inventory.csv`: source-network inventory and benchmark-membership flags.
- `real_network_metrics.csv`: descriptive and structural metrics for every network and projection.
- `real_network_plausibility_ranges.csv`: five-metric central 95% intervals and medians for the primary and sensitivity benchmarks under equal-study and equal-network weighting.
- `Fig_real_network_metrics.{pdf,png}`: primary Mutual benchmark.
- `Fig_real_network_metrics_OR_sensitivity.{pdf,png}`: OR-projection sensitivity.
- `Fig_real_network_benchmark_sensitivity.{pdf,png}`: benchmark-definition sensitivity.
- `Fig_normalized_degree_distribution_real_networks.csv`: shared 50-bin degree-distribution data, including benchmark flags.
- `Fig_normalized_degree_distribution_real_networks_{Mutual,OR}.{pdf,png}`: per-dataset CCDFs.
- `Fig_normalized_degree_histogram_real_networks_{Mutual,OR}.{pdf,png}`: per-dataset 25-bin histograms.
- `Fig_normalized_degree_distribution_real_networks_all_datasets.{pdf,png}`: dataset-level curves.
- `Fig_normalized_degree_histogram_real_networks_pooled.{pdf,png,csv}`: primary pooled reference with equal study weight; the CSV retains the 50-bin grid required by notebook 02.
